<a href="https://colab.research.google.com/github/R341CUL/python-together/blob/main/08-machine-learning/04_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4 · k-Nearest Neighbours — the simplest classifier

KNN is machine learning with almost no maths, and it's a great first classifier:
**to label a new point, look at the `k` closest known points and take a vote.**
"You are like your neighbours." We'll build it from scratch, then use sklearn.

In [ ]:
import numpy as np

## Some labelled data (two blobs)

Two groups of points: class 0 clustered low-left, class 1 clustered high-right.

In [ ]:
rng = np.random.default_rng(1)
class0 = rng.normal(loc=[1, 1], scale=0.6, size=(20, 2))
class1 = rng.normal(loc=[4, 4], scale=0.6, size=(20, 2))

X = np.vstack([class0, class1])
y = np.array([0] * 20 + [1] * 20)
print("X shape:", X.shape, " labels:", y)

## KNN from scratch

The whole algorithm, in a few lines:
1. measure the distance from the new point to every known point,
2. find the `k` smallest distances,
3. return the majority label among those `k` neighbours.

In [ ]:
def knn_predict(new_point, X, y, k=3):
    distances = np.sqrt(((X - new_point) ** 2).sum(axis=1))   # euclidean distance
    nearest = np.argsort(distances)[:k]                        # indices of k closest
    votes = y[nearest]
    return int(np.round(votes.mean()))                         # majority vote (0/1)

test_point = np.array([3.5, 3.8])
print("point", test_point, "-> class", knn_predict(test_point, X, y, k=3))

test_point2 = np.array([1.2, 0.8])
print("point", test_point2, "-> class", knn_predict(test_point2, X, y, k=3))

## The same thing with scikit-learn

In practice you use the library — it's faster and handles the edge cases. Notice
the shape of real ML: `train_test_split` → `fit` → `predict` → score.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, preds), 3))

## Choosing `k`

- small `k` (like 1) → sensitive to noise, can **overfit**.
- large `k` → smoother but can blur the boundary (**underfit**).

Try a few and see.

In [ ]:
for k in [1, 3, 5, 9]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"k={k}: accuracy {acc:.3f}")

## 🏋️ Your turn

Our from-scratch `knn_predict` uses majority vote but ignores *how far* each
neighbour is. Change it so `k` is a parameter you pass, then classify the point
`[2.5, 2.5]` (right in the middle) with `k=1` and `k=11` — does the answer flip?

In [ ]:
middle = np.array([2.5, 2.5])
print("k=1 :", knn_predict(middle, X, y, k=1))
print("k=11:", knn_predict(middle, X, y, k=11))

### ✅ What you learned
- KNN classifies by a **majority vote of the nearest neighbours** — no training loop.
- Distance (here Euclidean) defines "nearest".
- `k` trades noise-sensitivity (small) against over-smoothing (large).
- Real ML has a repeatable shape: **split → fit → predict → score**.

**Next:** `05_sklearn_pipeline.py` — doing it properly with scaling + a pipeline.